# Product 상세 정보 수집

이전 단계에서 수집한 Product ID 리스트를 사용하여 각 상품의 상세 정보를 수집합니다.

## Parameters

In [ ]:
# Papermill parameters (이전 단계에서 전달받음)
collected_product_ids = []
collection_summary = {}
version_date = None
limit = None

## 1. 라이브러리 및 설정 로드

In [ ]:
import requests
import json
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import sys
import os
from pathlib import Path

# Add parent directory to path for config import
sys.path.append('..')
from config import config

# 로깅 설정
logging.basicConfig(level=getattr(logging, config.log_level.upper()))
logger = logging.getLogger(__name__)

print(f"이전 단계에서 받은 Product ID 개수: {len(collected_product_ids)}")
print(f"이전 단계 요약: {collection_summary}")
print(f"Version Date: {version_date}")

## 2. 상세 정보 수집 함수 정의

In [ ]:
def fetch_product_detail(product_id: str, headers: Dict[str, str]) -> Optional[Dict[str, Any]]:
    """단일 상품의 상세 정보를 가져오는 함수"""
    
    url = f"{config.map_api_base_url}/product-meta/basic-plan_847de97c-96c0-49b1-b3b1-900ddc587e54/product-info"
    params = {"legacyId": product_id}
    
    try:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=config.api_timeout
        )
        
        if response.status_code == 200:
            return response.json()
        else:
            logger.warning(f"Product {product_id} - HTTP {response.status_code}: {response.text}")
            return None
            
    except requests.exceptions.RequestException as e:
        logger.error(f"Product {product_id} - 요청 실패: {str(e)}")
        return None

# API 헤더 설정
headers = {
    "Content-Type": "application/json",
    "x-apim-key": config.map_api_key
}

print("상세 정보 수집 함수 준비 완료")

## 3. 병렬로 상세 정보 수집 시작

In [ ]:
if not collected_product_ids:
    raise ValueError("이전 단계에서 Product ID가 전달되지 않았습니다.")

# 수집할 상품 수 제한 (limit 파라미터 적용)
products_to_collect = collected_product_ids
if limit and limit < len(products_to_collect):
    products_to_collect = products_to_collect[:limit]
    print(f"제한 적용: {len(products_to_collect)}개 상품만 수집")

print(f"총 {len(products_to_collect)}개 상품 상세 정보 수집 시작")
start_time = datetime.now()

# 결과 저장 변수
successful_results = []
failed_products = []
processed_count = 0

# ThreadPoolExecutor로 병렬 처리
with ThreadPoolExecutor(max_workers=config.max_workers) as executor:
    # 모든 작업을 제출
    future_to_product = {
        executor.submit(fetch_product_detail, product_id, headers): product_id 
        for product_id in products_to_collect
    }
    
    # 완료되는 대로 결과 처리
    for future in as_completed(future_to_product):
        product_id = future_to_product[future]
        processed_count += 1
        
        try:
            result = future.result()
            if result:
                successful_results.append(result)
            else:
                failed_products.append(product_id)
                
        except Exception as e:
            logger.error(f"Product {product_id} - 예외 발생: {str(e)}")
            failed_products.append(product_id)
        
        # 진행 상황 출력 (매 100개 또는 10%마다)
        if processed_count % 100 == 0 or processed_count % max(1, len(products_to_collect) // 10) == 0:
            progress = (processed_count / len(products_to_collect)) * 100
            print(f"진행률: {processed_count}/{len(products_to_collect)} ({progress:.1f}%) - "
                  f"성공: {len(successful_results)}, 실패: {len(failed_products)}")

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n수집 완료! 소요시간: {duration:.2f}초")
print(f"성공: {len(successful_results)}개, 실패: {len(failed_products)}개")

## 4. 결과 분석 및 요약

In [ ]:
# 성공률 계산
total_attempted = len(products_to_collect)
success_count = len(successful_results)
failure_count = len(failed_products)
success_rate = (success_count / total_attempted) * 100 if total_attempted > 0 else 0

# 수집 결과 요약
collection_result = {
    "timestamp": datetime.now().isoformat(),
    "version_date": version_date,
    "collection_stats": {
        "total_product_ids_received": len(collected_product_ids),
        "total_attempted": total_attempted,
        "successful_collections": success_count,
        "failed_collections": failure_count,
        "success_rate_percent": round(success_rate, 2),
        "duration_seconds": round(duration, 2)
    },
    "failed_product_ids": failed_products[:10] if failed_products else []  # 처음 10개만 기록
}

print("\n=== 상세 정보 수집 결과 요약 ===")
for key, value in collection_result["collection_stats"].items():
    print(f"{key}: {value}")

if failed_products:
    print(f"\n실패한 Product ID (처음 10개): {failed_products[:10]}")
    
# 수집된 데이터 샘플 확인
if successful_results:
    print(f"\n첫 번째 수집 데이터 구조: {list(successful_results[0].keys())}")
    
    # managementInfo가 있는지 확인 (다음 단계 비교용)
    if 'managementInfo' in successful_results[0]:
        mgmt_info = successful_results[0]['managementInfo']
        print(f"managementInfo 구조: {list(mgmt_info.keys()) if isinstance(mgmt_info, dict) else type(mgmt_info)}")
        
        if isinstance(mgmt_info, dict) and 'mappedProductCode' in mgmt_info:
            mapped_code = mgmt_info['mappedProductCode']
            print(f"mappedProductCode 구조: {list(mapped_code.keys()) if isinstance(mapped_code, dict) else type(mapped_code)}")

## 5. 파일 저장 (mobile_plan_info_YYYYMMDD.json)

In [ ]:
# 저장 디렉토리 생성
dataload_dir = Path(config.dataload_dir)
dataload_dir.mkdir(parents=True, exist_ok=True)

# 파일명 생성
if not version_date:
    version_date = datetime.now().strftime("%Y%m%d")

filename = f"mobile_plan_info_{version_date}.json"
file_path = dataload_dir / filename

# 메타데이터와 함께 저장할 데이터 구성
final_data = {
    "metadata": {
        "created_at": datetime.now().isoformat(),
        "version_date": version_date,
        "total_products": success_count,
        "success_rate": success_rate,
        "collection_duration_seconds": duration,
        "source": "MAP API - product-info endpoint"
    },
    "result_list": successful_results  # 기존 형식과 동일하게 result_list로 저장
}

# JSON 파일로 저장
try:
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n파일 저장 완료: {file_path}")
    print(f"파일 크기: {file_path.stat().st_size / (1024*1024):.2f} MB")
    
except Exception as e:
    logger.error(f"파일 저장 실패: {str(e)}")
    raise

# 다음 단계로 전달할 변수들
saved_file_path = str(file_path)
product_collection_result = collection_result
total_products_collected = success_count

print(f"\n다음 단계로 전달할 정보:")
print(f"- 저장된 파일: {saved_file_path}")
print(f"- 수집된 상품 수: {total_products_collected}")
print(f"- 성공률: {success_rate:.2f}%")

## 완료

상품 상세 정보 수집이 완료되었습니다. 다음 단계에서 이전 데이터와 비교하여 변경사항을 감지합니다.